# GenAI Phishing Detector — Evaluation & SHAP Explainability

Evaluate model performance and interpret predictions using SHAP.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn pandas numpy matplotlib seaborn shap

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add project to path
sys.path.insert(0, '/content/drive/MyDrive/genai-phishing-detector')

# Or clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/genai-phishing-detector.git /content/project
# sys.path.insert(0, '/content/project')

## Load Model and Test Data

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load model
model_path = '/content/drive/MyDrive/genai-phishing-detector/models/phishing_model'
tokenizer_path = '/content/drive/MyDrive/genai-phishing-detector/models/tokenizer'

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained(tokenizer_path)
model.to(device)
model.eval()
print('Model loaded successfully')

In [ ]:
# Load test data
csv_path = '/content/drive/MyDrive/genai-phishing-detector/dataset/dataset.csv'
df = pd.read_csv(csv_path).dropna().reset_index(drop=True)

# Or use the last 10% as test
from sklearn.model_selection import train_test_split
_, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['label'])
print(f'Test samples: {len(test_df)}')
print(test_df['label'].value_counts().sort_index())

## Run Inference

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

MAX_LENGTH = 128
BATCH_SIZE = 16

encodings = tokenizer(
    test_df['text'].tolist(),
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
    return_tensors='pt',
)

test_dataset = TensorDataset(
    encodings['input_ids'],
    encodings['attention_mask'],
    torch.tensor(test_df['label'].values)
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

## Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ['Legitimate', 'Traditional Phishing', 'AI-Generated Phishing']

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)

print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 Score:  {f1:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## ROC Curves

In [ ]:
y_bin = label_binarize(all_labels, classes=[0, 1, 2])
plt.figure(figsize=(10, 8))

for i in range(3):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Precision-Recall Curves

In [ ]:
plt.figure(figsize=(10, 8))

for i in range(3):
    prec, rec, _ = precision_recall_curve(y_bin[:, i], all_probs[:, i])
    ap = average_precision_score(y_bin[:, i], all_probs[:, i])
    plt.plot(rec, prec, label=f'{CLASS_NAMES[i]} (AP = {ap:.3f})', linewidth=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## SHAP Explainability

In [ ]:
import shap

# Define prediction function for SHAP
def predict_proba(texts):
    cleaned = [t.lower().strip() for t in texts]
    encodings = tokenizer(cleaned, truncation=True, padding='max_length',
                          max_length=128, return_tensors='pt')
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

# Test samples
test_texts = test_df['text'].iloc[:3].tolist()

# SHAP explainer (Partition algorithm for Transformers)
masker = shap.maskers.Text(tokenizer, mask_token='...', collapse_mask_token=True)
explainer = shap.Explainer(predict_proba, masker, output_names=CLASS_NAMES)

shap_values = explainer(test_texts)
print('SHAP values computed')

In [ ]:
# Visualize
for i in range(len(test_texts)):
    pred_idx = int(np.argmax(predict_proba([test_texts[i]])[0]))
    print(f'\nSample {i+1} — Prediction: {CLASS_NAMES[pred_idx]}')
    shap.plots.text(shap_values[i, :, pred_idx], display=False)
    plt.show()

In [ ]:
# Get top words for a specific sample
sample_idx = 0
pred_idx = int(np.argmax(predict_proba([test_texts[sample_idx]])[0]))
sv = shap_values[sample_idx, :, pred_idx]

word_imp = [(w, v) for w, v in zip(sv.data, sv.values) if w.strip() and w.strip() not in ('...',)]
word_imp.sort(key=lambda x: abs(x[1]), reverse=True)

print(f'Top influential words (prediction: {CLASS_NAMES[pred_idx]}):')
for word, imp in word_imp[:10]:
    print(f'  {word:20s} {imp:+.4f}')

---
**Next Steps:**
- Deploy model to HuggingFace Spaces
- Launch the Streamlit app
- Run batch inference on new data